# Movie Preprocess Content Update Frame Extractor

`data/movie_preprocess` 配下の `front` / `rear` 動画を別々に処理し、30fpsキャプチャ由来の重複フレームを間引いた動画を `outputs` 配下へ出力します。

各動画では隣接フレーム差分から内容更新フレームを抽出し、その内容更新フレーム列でgrid flow特徴量も算出します。可視化描画やnear-zero flowリストアップは行いません。


## 処理の流れ

1. `data/movie_preprocess` 配下のfront/rear動画を一覧化します。
2. 内容更新フレームを選び、間引き動画とgrid flow特徴を作ります。
3. summary CSVと特徴CSVを `outputs/movie_preprocess_content_update_frames` に保存します。


In [ ]:
# ------------------------------------------------------------
# セル概要: ライブラリ読み込みと処理設定
# ------------------------------------------------------------
# - data/movie_preprocess のfront/rear動画を読み、内容更新フレームだけを残すための共通設定をまとめます。
# - 内容更新判定とgrid flow集計は src/forklift_features の共通関数を使います。
# ------------------------------------------------------------

from __future__ import annotations

from pathlib import Path
from typing import Any

import importlib
import sys

import cv2
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(value: Any) -> None:
        print(value)

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        fallback_iterable = [] if iterable is None else iterable
        return fallback_iterable

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "movie_preprocess").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repository root from {start}")


PROJECT_ROOT = find_project_root()
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from forklift_features import content_update, flow_extract
content_update = importlib.reload(content_update)
flow_extract = importlib.reload(flow_extract)

MOVIE_PREPROCESS_DIR = PROJECT_ROOT / "data" / "movie_preprocess"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "movie_preprocess_content_update_frames"
THINNED_VIDEO_DIR = OUTPUT_DIR / "thinned_movie_preprocess"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
THINNED_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

PROCESS_VIEWS = ("front", "rear")
MAX_VIDEOS: int | None = None
OVERWRITE_THINNED_VIDEO = False
VIDEO_FOURCC = "mp4v"

DEDUP_ENABLED = True
DUPLICATE_DIFF_THRESHOLD = content_update.DEFAULT_CONTENT_UPDATE_DIFF_THRESHOLD
FLOW_TARGET_DT = content_update.DEFAULT_FLOW_TARGET_DT
STATIC_SCENE_MIN_SEC = content_update.DEFAULT_STATIC_SCENE_MIN_SEC
FLOW_NORMALIZE_BY_DT = content_update.DEFAULT_FLOW_NORMALIZE_BY_DT

ANALYSIS_RESIZE_WIDTH = content_update.DEFAULT_CONTENT_UPDATE_RESIZE_WIDTH
ANALYSIS_GAUSSIAN_BLUR_KERNEL: int | None = content_update.DEFAULT_CONTENT_UPDATE_GAUSSIAN_BLUR_KERNEL
FLOW_GRID = (3, 3)
FARNEBACK_PARAMS = dict(flow_extract.DEFAULT_FARNEBACK_PARAMS)

print(f"PROJECT_ROOT         : {PROJECT_ROOT}")
print(f"MOVIE_PREPROCESS_DIR : {MOVIE_PREPROCESS_DIR}")
print(f"OUTPUT_DIR           : {OUTPUT_DIR}")
print(f"THINNED_VIDEO_DIR    : {THINNED_VIDEO_DIR}")
print(f"PROCESS_VIEWS        : {PROCESS_VIEWS}")
print(f"DEDUP_ENABLED        : {DEDUP_ENABLED}")
print(f"DUPLICATE_DIFF_THRESHOLD: {DUPLICATE_DIFF_THRESHOLD}")
print(f"FLOW_TARGET_DT       : {FLOW_TARGET_DT}")
print(f"FLOW_NORMALIZE_BY_DT : {FLOW_NORMALIZE_BY_DT}")


In [ ]:
# ------------------------------------------------------------
# セル概要: 入力動画の一覧化と出力先決定
# ------------------------------------------------------------
# - ファイルパスから sample_id / view / category / environment を推定します。
# - 間引き後動画は元のcategory/environment構造を保って outputs 配下へ保存します。
# ------------------------------------------------------------

def infer_video_identity(path: Path, movie_preprocess_dir: Path = MOVIE_PREPROCESS_DIR) -> dict[str, Any]:
    try:
        category, environment = path.relative_to(movie_preprocess_dir).parts[:2]
    except Exception:
        category, environment = "unknown", "unknown"
    stem = path.stem
    if stem.endswith("_front"):
        sample_id = stem[:-len("_front")]
        view = "front"
    elif stem.endswith("_rear"):
        sample_id = stem[:-len("_rear")]
        view = "rear"
    else:
        sample_id = stem
        view = "unknown"
    return {
        "sample_id": sample_id,
        "view": view,
        "category": str(category),
        "environment": str(environment),
        "video_path": path,
    }


def movie_sort_key(row: dict[str, Any]) -> tuple[int, str, str, str]:
    category_rank = 0 if str(row.get("category")) == "normal" else 1
    return category_rank, str(row.get("sample_id")), str(row.get("view")), Path(row.get("video_path")).name


def list_movie_preprocess_videos(movie_preprocess_dir: Path = MOVIE_PREPROCESS_DIR) -> pd.DataFrame:
    rows = []
    for path in sorted(movie_preprocess_dir.glob("**/*.mp4"), key=lambda value: value.name):
        if not path.is_file():
            continue
        row = infer_video_identity(path, movie_preprocess_dir)
        if row["view"] not in PROCESS_VIEWS:
            continue
        rows.append(row)
    rows = sorted(rows, key=movie_sort_key)
    return pd.DataFrame(rows, columns=["sample_id", "view", "category", "environment", "video_path"])



def thinned_output_path(video_path: Path) -> Path:
    try:
        relative_path = video_path.resolve().relative_to(MOVIE_PREPROCESS_DIR.resolve())
    except Exception:
        relative_path = Path(video_path.name)
    return THINNED_VIDEO_DIR / relative_path


In [ ]:
# ------------------------------------------------------------
# セル概要: 内容更新フレーム抽出とgrid flow特徴作成
# ------------------------------------------------------------
# - 隣接フレーム差分で重複キャプチャフレームを飛ばし、内容が変わったフレームだけを残します。
# - 残したフレーム間でFarneback flowを計算し、共通grid flow集計で3x3特徴に変換します。
# ------------------------------------------------------------

def extract_content_update_flow(video_row: pd.Series) -> tuple[dict[str, Any], pd.DataFrame]:
    video_path = Path(video_row["video_path"])
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        return {"status": "open_failed", "video_path": str(video_path)}, pd.DataFrame()

    fps = content_update.safe_video_fps(capture)
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration_sec = float(frame_count / fps) if frame_count > 0 else 0.0
    content_frame_indices: list[int] = []
    feature_rows: list[dict[str, Any]] = []

    previous_adjacent_gray: np.ndarray | None = None
    previous_content_gray: np.ndarray | None = None
    previous_content_time: float | None = None
    duplicate_count_since_last_update = 0
    duplicate_frame_count = 0
    read_failed_count = 0
    flow_failed_count = 0
    max_time_since_content_update = 0.0
    content_frame_index = -1
    frame_index = 0

    try:
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            time_sec = float(frame_index / fps)
            try:
                gray = content_update.prepare_gray_frame(
                    frame,
                    resize_width=ANALYSIS_RESIZE_WIDTH,
                    gaussian_blur_kernel=ANALYSIS_GAUSSIAN_BLUR_KERNEL,
                )
            except Exception:
                read_failed_count += 1
                frame_index += 1
                continue

            if previous_adjacent_gray is None:
                frame_diff = np.nan
                is_duplicate = False
            else:
                frame_diff = content_update.frame_mean_abs_diff(gray, previous_adjacent_gray)
                is_duplicate = bool(DEDUP_ENABLED and not content_update.is_content_update(frame_diff, diff_threshold=DUPLICATE_DIFF_THRESHOLD))

            if is_duplicate:
                duplicate_frame_count += 1
                duplicate_count_since_last_update += 1
                if previous_content_time is not None:
                    max_time_since_content_update = max(max_time_since_content_update, time_sec - previous_content_time)
                previous_adjacent_gray = gray
                frame_index += 1
                continue

            content_frame_index += 1
            content_frame_indices.append(int(frame_index))
            if previous_content_gray is not None and previous_content_time is not None:
                source_dt = max(float(time_sec - previous_content_time), 1e-6)
                time_since_last_content_update = source_dt
                max_time_since_content_update = max(max_time_since_content_update, time_since_last_content_update)
                static_scene_flag = int(time_since_last_content_update >= float(STATIC_SCENE_MIN_SEC))
                try:
                    flow = cv2.calcOpticalFlowFarneback(previous_content_gray, gray, None, **FARNEBACK_PARAMS)
                    if FLOW_NORMALIZE_BY_DT:
                        flow = flow * (float(FLOW_TARGET_DT) / source_dt)
                    # 共通実装で3x3セルへ集約し、このNotebook固有の診断列だけを後段で付与します。
                    grid_rows = flow_extract.compute_grid_flow_features(
                        flow,
                        frame_index=frame_index,
                        time_sec=time_sec,
                        grid=FLOW_GRID,
                        sample_index=content_frame_index,
                        source_dt=source_dt,
                    )
                    feature_rows.extend({
                        "video_id": str(video_row["sample_id"]),
                        "view": str(video_row["view"]),
                        "time": float(row["time_sec"]),
                        "content_frame_index": int(content_frame_index),
                        "source_frame_index": int(row["frame_index"]),
                        "grid_y": int(row["grid_y"]),
                        "grid_x": int(row["grid_x"]),
                        "flow_x": float(row["flow_dx_mean"]),
                        "flow_y": float(row["flow_dy_mean"]),
                        "flow_mag_mean": float(row["flow_mag_mean"]),
                        "flow_vector_mag": float(row["flow_vector_mag"]),
                        "source_dt": float(row["source_dt"]),
                        "frame_diff": float(frame_diff),
                        "is_duplicate": 0,
                        "is_content_update": 1,
                        "static_scene_flag": int(static_scene_flag),
                        "duplicate_count_since_last_update": int(duplicate_count_since_last_update),
                        "time_since_last_content_update": float(time_since_last_content_update),
                    } for row in grid_rows)
                except Exception:
                    flow_failed_count += 1

            previous_content_gray = gray
            previous_content_time = time_sec
            previous_adjacent_gray = gray
            duplicate_count_since_last_update = 0
            frame_index += 1
    finally:
        capture.release()

    content_frame_count = int(len(content_frame_indices))
    content_update_rate_fps = float(content_frame_count / duration_sec) if duration_sec > 0.0 and content_frame_count > 0 else np.nan
    output_fps = content_update.content_update_output_fps(content_frame_count, duration_sec, fps)
    output_path = thinned_output_path(video_path)
    video_write_info = content_update.write_selected_frames_mp4(
        video_path,
        output_path,
        content_frame_indices,
        output_fps=output_fps,
        fourcc_name=VIDEO_FOURCC,
        overwrite=OVERWRITE_THINNED_VIDEO,
    )

    status = "ok"
    warning = ""
    if content_frame_count < 2:
        status = "too_few_content_frames"
        warning = "Content update frames are too few for flow calculation."

    summary = {
        "status": status,
        "warning": warning,
        "sample_id": str(video_row["sample_id"]),
        "view": str(video_row["view"]),
        "category": str(video_row["category"]),
        "environment": str(video_row["environment"]),
        "video_path": str(video_path),
        "fps": float(fps),
        "frame_count": int(frame_count),
        "duration_sec": float(duration_sec),
        "content_frame_count": content_frame_count,
        "duplicate_frame_count": int(duplicate_frame_count),
        "content_frame_ratio": float(content_frame_count / frame_count) if frame_count > 0 else 0.0,
        "content_update_rate_fps": content_update_rate_fps,
        "max_time_since_content_update": float(max_time_since_content_update),
        "trailing_static_scene_flag": int(max_time_since_content_update >= float(STATIC_SCENE_MIN_SEC)),
        "read_failed_count": int(read_failed_count),
        "flow_failed_count": int(flow_failed_count),
        "feature_rows": int(len(feature_rows)),
        **video_write_info,
    }
    return summary, pd.DataFrame(feature_rows)


In [ ]:
# ------------------------------------------------------------
# セル概要: 全対象動画の実行とCSV保存用テーブル作成
# ------------------------------------------------------------
# - 対象動画を順に処理し、summary行とgrid flow特徴行をDataFrameへ集約します。
# - 実ファイルの保存は extract_content_update_flow 内で行い、ここでは結果を確認表示します。
# ------------------------------------------------------------

video_df = list_movie_preprocess_videos()
if MAX_VIDEOS is not None:
    video_df = video_df.head(int(MAX_VIDEOS)).copy()

print(f"target videos: {len(video_df)}")
display(video_df.head(20))

summary_rows: list[dict[str, Any]] = []
feature_parts: list[pd.DataFrame] = []
for video_row in tqdm(list(video_df.itertuples(index=False)), desc="extract content update videos"):
    row = pd.Series(video_row._asdict())
    summary, features_part = extract_content_update_flow(row)
    summary_rows.append(summary)
    if len(features_part):
        features_part.insert(0, "category", summary.get("category", "unknown"))
        features_part.insert(1, "environment", summary.get("environment", "unknown"))
        features_part.insert(2, "source_video_path", summary.get("video_path"))
        features_part.insert(3, "thinned_video_path", summary.get("thinned_video_path"))
        feature_parts.append(features_part)

summary_df = pd.DataFrame(summary_rows)
features_df = pd.concat(feature_parts, ignore_index=True) if feature_parts else pd.DataFrame()

print(f"processed videos: {len(summary_df)}")
print(f"thinned videos output under: {THINNED_VIDEO_DIR}")
display(summary_df)
display(features_df.head(50))

if len(summary_df) and summary_df["status"].ne("ok").any():
    print("Warnings / non-ok videos:")
    display(summary_df.loc[summary_df["status"].ne("ok"), ["sample_id", "view", "status", "warning", "video_path"]])
